# 01 - Pulizia dei dati
In questo notebook carichiamo il dataset e controlliamo che sia pulito

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/demand.csv")
df.head() #mostra le prime 5 righe, per vedere com'è fatta la tabella

,Product_Code,Warehouse,Product_Category,Date,Order_Demand
0,Product_0993,Whse_J,Category_028,2012/7/27,100
1,Product_0979,Whse_J,Category_028,2012/1/19,500
2,Product_0979,Whse_J,Category_028,2012/2/3,500
3,Product_0979,Whse_J,Category_028,2012/2/9,500
4,Product_0979,Whse_J,Category_028,2012/3/2,500


In [2]:
df.shape #numero di righe e colonne

(1048575, 5)

In [3]:
df.info() #mostra per ogni colonna quante righe non sono vuote e il tipo di dato

<class 'pandas.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 5 columns):
 #   Column            Non-Null Count    Dtype
---  ------            --------------    -----
 0   Product_Code      1048575 non-null  str  
 1   Warehouse         1048575 non-null  str  
 2   Product_Category  1048575 non-null  str  
 3   Date              1037336 non-null  str  
 4   Order_Demand      1048575 non-null  str  
dtypes: str(5)
memory usage: 40.0 MB


In [4]:
df.isna().sum() #quanti valori mancanti

Product_Code            0
Warehouse               0
Product_Category        0
Date                11239
Order_Demand            0
dtype: int64

## Nota sui valori mancanti nella colonna Date

**Cosa abbiamo trovato:** 11.239 righe (circa l'1% del dataset) non hanno la data.

**Perché potrebbero mancare le date (ipotesi):**
- Errore di inserimento: l'operatore non ha compilato il campo.
- Problema di sistema o di importazione: l'ERP o il file di esportazione ha perso il dato.
- Ordini speciali o in attesa di conferma, registrati prima di avere una data di consegna.
- Un magazzino o un processo che non registra la data in modo sistematico.
- Anonimizzazione o pulizia del dataset da parte di chi lo ha pubblicato.

**Non sappiamo quindi se queste righe siano errori, duplicati o ordini reali.**

**Decisione presa:** non cancelliamo e non inventiamo nessuna data.
Una data "indicativa" falserebbe la serie storica e quindi la previsione.

**Come si potrebbe risolvere:**
1. Chiedere a chi gestisce l'ERP o il sistema di origine perché mancano.
2. Recuperare la data da un'altra fonte (ordine di acquisto, bolla, fattura) usando un codice comune.
3. Se la data è davvero persa, tenere le righe da parte e usarle solo per analisi che non
   dipendono dal tempo (es. volume totale per prodotto).
4. Se sono errori accertati, correggerli alla fonte, così non si ripetono.

**Come le trattiamo nel progetto:** le escludiamo dalla previsione nel tempo, che richiede la data,
e le segnaliamo nel README come limite dei dati.

In [5]:
df.dtypes #se Order_demand è "object", pandas lo sta trattando come TESTO, non come numero

Product_Code        str
Warehouse           str
Product_Category    str
Date                str
Order_Demand        str
dtype: object

In [6]:
#controllo se alcuni valori hanno le parentesi, es. "(5)", che nella notazione contabile indicano un numero NEGATIVO (es. un reso).
print(df[df["Order_Demand"].astype(str).str.contains("(", regex=False)].head(10))

        Product_Code Warehouse Product_Category        Date Order_Demand
112290  Product_2169    Whse_A     Category_024    2012/8/9          (1)
112307  Product_2132    Whse_A     Category_009   2012/11/1         (24)
112308  Product_2144    Whse_A     Category_009   2012/11/1         (24)
112356  Product_2118    Whse_A     Category_009    2012/3/7         (50)
112357  Product_2120    Whse_A     Category_009    2012/3/7        (100)
112360  Product_1794    Whse_A     Category_024   2012/6/28          (1)
112378  Product_1461    Whse_A     Category_019   2012/5/24        (150)
112441  Product_0935    Whse_A     Category_006   2012/7/19       (1000)
112563  Product_1544    Whse_A     Category_019  2012/10/12       (2500)
112610  Product_0982    Whse_A     Category_028   2012/7/19       (5000)


In [7]:
#controlliamo quanti sono
print("Righe con parentesi:", df["Order_Demand"].astype(str).str.contains("(", regex=False).sum())

Righe con parentesi: 10469


In [8]:
# Per convertire Order_Demand in numero:

df["Order_Demand"] = (
    df["Order_Demand"]
    .astype(str) 
    .str.strip() 
    .str.replace("(", "-", regex=False) #sostituiamo "(" con "-" così "(5" diventa "-5"
    .str.replace(")", "", regex=False) #togliamo la ")" finale
)

df["Order_Demand"] = pd.to_numeric(df["Order_Demand"]) #trasformo testo in numero

In [9]:
print(df["Order_Demand"].dtype) #controllo della cella

int64


In [10]:
df["Order_Demand"].describe() #statistiche di base

count    1.048575e+06
mean     4.867427e+03
std      2.893346e+04
min     -9.990000e+05
25%      1.800000e+01
50%      3.000000e+02
75%      2.000000e+03
max      4.000000e+06
Name: Order_Demand, dtype: float64

In [ ]:
#Per controllo finale: quanti valori sono negativi (le ex parentesi)?
print("Valori negativi:", (df["Order_Demand"] < 0).sum())

Valori negativi: 10469


## Nota su Order_Demand: conversione in numero

**Problema:** la colonna Order_Demand viene letta come testo e non come numero.

**Causa trovata:** 10.469 righe (circa l'1% del dataset) hanno il valore scritto tra parentesi,
per esempio (1), (24), (100), (5000). Per questo pandas non riconosce la colonna come numerica.

**Ipotesi:** le parentesi seguono la notazione contabile e indicano un valore negativo
(per esempio un reso o una rettifica). Il dataset non lo conferma: è un'assunzione da verificare.

**Perché dobbiamo convertirla:** con il testo non si possono fare somme, medie, grafici né previsioni.

**Cosa facciamo:** sostituiamo "(" con "-" e togliamo ")", così "(24)" diventa -24,
poi convertiamo la colonna in numero. Il file originale demand.csv non viene modificato.

**Controllo:** la conversione è riuscita senza errori e la colonna è ora di tipo int64.
Il valore minimo è -999.000: un valore molto grande, da controllare tra gli outlier.

**In un caso aziendale reale:** chiederemmo a chi gestisce l'ERP cosa indicano le parentesi,
prima di fidarci di questa interpretazione, e se i resi vanno sottratti dalla domanda o trattati a parte.